# Lesson 6.2 — Metrics Query Type in Grafana

## Learning Objectives
- Build Metrics queries using the AppDynamics Grafana query editor
- Navigate the metric hierarchy to find specific metric paths
- Configure aggregation, roll-up, and label options
- Build multi-metric panels comparing multiple BTs or tiers

**Reference**: [AppDynamics Query Editor](https://grafana.com/docs/plugins/dlopes7-appdynamics-datasource/latest/query-editor/) | [Metrics Query Type](https://grafana.com/docs/plugins/dlopes7-appdynamics-datasource/latest/query-editor/#metrics-query-type)

## 1. The Grafana AppDynamics Query Editor

The query editor appears when you edit a panel and select the AppDynamics datasource.

### Query type selector:
At the top of the query editor, radio buttons let you select:
```
○ Metrics  ● Analytics  ○ Health  ○ Events  ○ Tiers  ○ Metric Names
```

### Metrics query type fields:

| Field | Description |
|-------|-------------|
| **Application** | Select from dropdown or enter `${application}` template variable |
| **Metric** | Segment-by-segment navigation through the metric hierarchy |
| **Aggregation** | Value, Sum, Min, Max, Count (default: Value) |
| **Roll-up** | Toggle: return only the latest single value (for Stat panels) |
| **Delimiter** | Character used to split metric path: `\|` or `:` |

### Raw edit mode:
Click the **pencil icon** next to the Metric field to switch to raw text input — type the full metric path directly. Click the checkmark to apply.

In [ ]:
# Reference: Label configuration for Grafana AppDynamics metrics queries
# From official docs: https://grafana.com/docs/plugins/dlopes7-appdynamics-datasource/latest/query-editor/

label_configurations = [
    {
        "mode": "Full Path",
        "description": "Uses the full metric path as the label",
        "example_metric": "Overall Application Performance|Average Response Time (ms)",
        "result_label": "Overall Application Performance|Average Response Time (ms)",
        "use_case": "Default; good for debugging; verbose for dashboards"
    },
    {
        "mode": "Segments",
        "description": "Builds label from specific segment indices (1-based)",
        "example_metric": "Errors|mywebsite|Error|Errors per Minute",
        "segments_selected": [2, 4],
        "result_label": "mywebsite|Errors per Minute",
        "use_case": "When you want a clean label from specific path segments"
    },
    {
        "mode": "Custom",
        "description": "Combines free text with aliasing patterns",
        "example_metric": "Overall Application Performance|Average Response Time (ms)",
        "template": "{{app}} MetricPart2: {{2}}",
        "app_name": "myApp",
        "result_label": "myApp MetricPart2: Average Response Time (ms)",
        "use_case": "Most flexible — custom text + metadata from metric path",
        "placeholders": {
            "{{app}}": "The application name",
            "{{n}}": "The nth segment from the metric path (1-based)"
        }
    },
]

print("=== Grafana AppDynamics Label Configuration Options ===")
print("Reference: https://grafana.com/docs/plugins/dlopes7-appdynamics-datasource/latest/query-editor/")
print()
for lc in label_configurations:
    print(f"Mode: {lc['mode']}")
    print(f"  Description: {lc['description']}")
    print(f"  Example metric: {lc['example_metric']}")
    if 'segments_selected' in lc:
        print(f"  Segments: {lc['segments_selected']}")
    if 'template' in lc:
        print(f"  Template: {lc['template']}")
    print(f"  Result: {lc['result_label']}")
    print(f"  Use case: {lc['use_case']}")
    print()

## 2. Common Grafana Panel Configurations

### Time Series panel — Response Time Trend
1. Add a new **Time Series** panel
2. Query type: **Metrics**
3. Application: `StoreFront`
4. Metric: navigate to `Business Transaction Performance|Business Transactions|WebFrontend|/checkout|Average Response Time (ms)`
5. Labels: **Custom** → `{{app}} - {{4}}` → produces `StoreFront - /checkout`
6. Click **+ Add query** and add `/search` metric
7. Y-axis: Units = milliseconds (ms)

### Stat panel — Current Calls per Minute
1. Add a **Stat** panel
2. Query type: **Metrics**
3. Application: `StoreFront`
4. Metric: `Overall Application Performance|Calls per Minute`
5. Aggregation: **Sum**
6. **Roll-up**: Toggle ON — returns a single value
7. Units: requests per minute (req/min)
8. Thresholds: 0-100 Green, 100-500 Yellow, >500 Red

### Gauge panel — Error Rate
1. Add a **Gauge** panel
2. Metric: `Overall Application Performance|Errors per Minute`
3. Roll-up: ON
4. Min: 0, Max: 50
5. Thresholds: Green 0-5, Yellow 5-20, Red 20+

In [ ]:
# Document Grafana panel configurations as code
# This can be used to provision panels via Grafana's dashboard JSON

grafana_panels = [
    {
        "title": "Application Response Time",
        "type": "timeseries",
        "datasource": "AppDynamics Production",
        "targets": [
            {
                "queryType": "Metrics",
                "application": "StoreFront",
                "metricPath": "Overall Application Performance|Average Response Time (ms)",
                "aggregation": "Value",
                "labelMode": "Custom",
                "labelTemplate": "{{app}} - Avg RT"
            }
        ],
        "fieldConfig": {
            "unit": "ms",
            "thresholds": [
                {"value": None, "color": "green"},
                {"value": 500, "color": "yellow"},
                {"value": 1000, "color": "red"}
            ]
        }
    },
    {
        "title": "Calls per Minute (Current)",
        "type": "stat",
        "datasource": "AppDynamics Production",
        "targets": [
            {
                "queryType": "Metrics",
                "application": "StoreFront",
                "metricPath": "Overall Application Performance|Calls per Minute",
                "aggregation": "Sum",
                "rollup": True,
                "labelMode": "Custom",
                "labelTemplate": "{{app}}"
            }
        ],
        "fieldConfig": {"unit": "reqps"}
    },
    {
        "title": "BT Comparison: /checkout vs /search",
        "type": "timeseries",
        "datasource": "AppDynamics Production",
        "targets": [
            {
                "queryType": "Metrics",
                "application": "StoreFront",
                "metricPath": "Business Transaction Performance|Business Transactions|WebFrontend|/checkout|Average Response Time (ms)",
                "labelMode": "Segments",
                "segments": [4]
            },
            {
                "queryType": "Metrics",
                "application": "StoreFront",
                "metricPath": "Business Transaction Performance|Business Transactions|WebFrontend|/search|Average Response Time (ms)",
                "labelMode": "Segments",
                "segments": [4]
            }
        ],
        "fieldConfig": {"unit": "ms"}
    }
]

print("=== Grafana Panel Configurations ===")
for panel in grafana_panels:
    print(f"\nPanel: [{panel['type'].upper()}] {panel['title']}")
    for i, t in enumerate(panel['targets'], 1):
        print(f"  Query {i}: {t['queryType']} — {t['application']} | {t['metricPath']}")
        if t.get('rollup'):
            print(f"    Roll-up: ON (single value)")
        if t.get('labelMode'):
            print(f"    Labels: {t['labelMode']}")

In [ ]:
# EXERCISE 6.2: Configure panels for the StoreFront Grafana dashboard
#
# TASK: For each panel requirement below, specify:
#   - Query type: Metrics
#   - Application name
#   - Full metric path
#   - Label configuration
#   - Aggregation and roll-up settings

panel_requirements = [
    {
        "requirement": "Show average response time for ALL BTs in WebFrontend tier (one line per BT)",
        "hint": "Use wildcard * for BT name",
        "metric_path": "???",  # TODO: include wildcard
        "label_mode": "???",   # TODO
        "rollup": False
    },
    {
        "requirement": "Show current JVM heap usage for order-svc-1 node (single stat)",
        "hint": "JVM|Heap|Current Usage (MB)",
        "metric_path": "???",
        "label_mode": "Custom",
        "label_template": "???",
        "rollup": True
    },
    {
        "requirement": "Show CPU % for all nodes in OrderService tier (multi-line chart)",
        "hint": "Use * for node name",
        "metric_path": "???",
        "label_mode": "Segments",
        "segments": ["???"],  # which segment(s) give a useful label?
        "rollup": False
    }
]

for req in panel_requirements:
    print(f"Requirement: {req['requirement']}")
    print(f"  Metric Path   : {req['metric_path']}")
    print(f"  Label Mode    : {req['label_mode']}")
    print(f"  Roll-up       : {req['rollup']}")
    print()

## Summary

- The **Metrics query type** queries AppDynamics metrics via the Controller REST API
- Navigate the metric hierarchy segment-by-segment, or switch to **raw edit mode** for direct path input
- **Aggregation options**: Value, Sum, Min, Max, Count
- **Roll-up**: ON returns a single data point (for Stat/Gauge panels); OFF returns a time series
- **Label modes**: Full Path, Segments (pick specific path segments), Custom (free text + `{{app}}` / `{{n}}`)
- Use `*` wildcards in metric paths to plot all matching metrics as separate series

---

**Next**: [Lesson 6.3 — Analytics ADQL Queries in Grafana](lesson-6.3-analytics-adql-in-grafana.ipynb)